In [ ]:
# Localiza la raíz del repositorio subiendo desde donde se ejecute el notebook,
# para no depender de una ruta fija de una máquina concreta.
from pathlib import Path

PROJECT_DIR = Path.cwd().resolve()
while not ((PROJECT_DIR / "data").exists() and (PROJECT_DIR / "notebooks").exists()):
    PROJECT_DIR = PROJECT_DIR.parent

# 1. Importación

Carga de las librerías necesarias y del dataset de features resultante de `02_feature_engineering.ipynb` (`data/processed/malaga/listings_full_features.csv`).

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [2]:
df = pd.read_csv(f"{PROJECT_DIR}/data/processed/malaga/listings_full_features.csv")
df.shape

(8742, 82)

## 2. Preparación de X e y

Separar identificadores, objetivo (`price`) y features. Convertir las columnas booleanas a 0/1 y decidir qué hacer con los nulos que quedan, ya que una regresión lineal no admite `NaN` directamente.

### 2.1 Identificadores, objetivo y features

Además de los identificadores, hay que excluir de `feature_cols` las columnas calculadas directamente a partir de `price` (`price_log`, `price_per_accommodate`, `price_per_min_night`): si se dejan dentro, el modelo no aprende ningún patrón real, deshace la fórmula y "adivina" el precio exacto. Es fuga de información, igual que en las otras tres ciudades.

**Fuga corregida más abajo**: `neighbourhood_price_encoded` (creada en `02_feature_engineering.ipynb`) se calculó allí usando todo el dataset, no solo lo que aquí es train. Se arregla en la sección 3.1bis, justo después del split. Por eso `neighbourhood_cleansed` (la columna cruda) sigue en `df` en este punto.

In [3]:
id_cols = ["id", "host_id", "host_profile_id"]
target_col = "price"
leakage_cols = ["price_log", "price_per_accommodate", "price_per_min_night"]
# neighbourhood_cleansed todavía no es una feature (pendiente de la corrección de la
# sección 3.1bis): se excluye de X igual que los identificadores, pero se mantiene en
# df para poder usarla justo después del split.
pending_cols = ["neighbourhood_cleansed"]
feature_cols = [c for c in df.columns if c not in id_cols + [target_col] + leakage_cols + pending_cols]

X = df[feature_cols].copy()
y = df[target_col].copy()

X.shape, y.shape

((8742, 74), (8742,))

### 2.2 Booleanas a 0/1

43 de las 74 features son booleanas (los one-hot y los flags). `scikit-learn` las admite tal cual, pero se convierten a `int` de forma explícita.

In [4]:
bool_cols = X.select_dtypes(include="bool").columns
X[bool_cols] = X[bool_cols].astype(int)
len(bool_cols)

43

### 2.3 Nulos restantes

`review_scores_rating`, `listing_age_days` y `days_since_last_review` son `NaN` en las 787 filas sin reviews todavía (`has_reviews == False`). Para este baseline se imputan con la mediana. Un modelo de árboles en `04_model_training.ipynb` podrá trabajar con el `NaN` directamente.

In [5]:
null_cols = X.columns[X.isnull().any()].tolist()
print(null_cols)

X[null_cols] = X[null_cols].fillna(X[null_cols].median())
X.isnull().sum().sum()

['review_scores_rating', 'listing_age_days', 'days_since_last_review']


np.int64(0)

`X` queda con 74 columnas numéricas sin nulos, e `y` es `price` sin transformar (el logaritmo se aplica más adelante, solo para el modelo de la sección 6).

## 3. Train/test split

Reservar un conjunto de test antes de tocar nada más, y guardarlo en `data/processed/malaga/` para que `04_model_training.ipynb` y `05_model_evaluation.ipynb` partan del mismo split.

### 3.1 Dividir

80/20, con `random_state` fijo. Se estratifica por `room_type`: en la EDA se vio que `Hotel room` (0.05%, 4 anuncios) y `Shared room` (0.38%, 33 anuncios) tienen muestra muy escasa en Málaga. Un split aleatorio sin más podría dejar a alguna de las dos con muy pocas filas en test por puro azar.

In [6]:
room_type_cols = [c for c in df.columns if c.startswith("room_type_")]
room_type_for_stratify = df[room_type_cols].idxmax(axis=1)

X_train, X_test, y_train, y_test, df_train, df_test = train_test_split(
    X, y, df, test_size=0.2, random_state=42, stratify=room_type_for_stratify
)

X_train.shape, X_test.shape

((6993, 74), (1749, 74))

### 3.1bis Corregir la fuga de `neighbourhood_price_encoded`

Mismo cálculo que en `02_feature_engineering.ipynb` (media de `price` por `neighbourhood_cleansed`, suavizada con la media global y `smoothing=10`), pero ahora solo con `df_train`. El mapa aprendido en train se aplica tal cual a `df_test` (un barrio de test que no apareciera en train recibiría la media global de train como respaldo).

Con esto corregido, `neighbourhood_cleansed` ya cumplió su función y se descarta de `df_train`/`df_test`. Como esta misma columna también generó los 11 dummies `district_*` en `02_feature_engineering.ipynb` (sección 5.2, al no existir un nivel de distrito separado en Málaga), esos dummies no se ven afectados por esta corrección: se calcularon directamente por fila, sin agregación sobre todo el dataset, así que no tenían fuga que corregir.

In [7]:
smoothing = 10
global_mean_price_train = y_train.mean()
neigh_stats_train = df_train.groupby("neighbourhood_cleansed")["price"].agg(["mean", "count"])
smoothed_mean_train = (
    neigh_stats_train["count"] * neigh_stats_train["mean"] + smoothing * global_mean_price_train
) / (neigh_stats_train["count"] + smoothing)

df_train["neighbourhood_price_encoded"] = df_train["neighbourhood_cleansed"].map(smoothed_mean_train)
df_test["neighbourhood_price_encoded"] = (
    df_test["neighbourhood_cleansed"].map(smoothed_mean_train).fillna(global_mean_price_train)
)

print("barrios de test no vistos en train:", df_test["neighbourhood_cleansed"].map(smoothed_mean_train).isnull().sum())

# X_train/X_test ya tenían la versión con fuga (calculada en la sección 2.1 antes del
# split): se sincronizan con el valor corregido de df_train/df_test.
X_train["neighbourhood_price_encoded"] = df_train["neighbourhood_price_encoded"]
X_test["neighbourhood_price_encoded"] = df_test["neighbourhood_price_encoded"]

df_train = df_train.drop(columns=["neighbourhood_cleansed"])
df_test = df_test.drop(columns=["neighbourhood_cleansed"])

df_train[["neighbourhood_price_encoded"]].describe()

barrios de test no vistos en train: 0


,neighbourhood_price_encoded
count,6993.000000
mean,212.837384
std,31.860574
min,155.166744
25%,207.995611
50%,207.995611
75%,207.995611
max,390.368879


Como se esperaba, los 11 distritos de Málaga aparecen todos en train (con `Centro` solo ya son 5902 anuncios de los 8742 totales), ningún distrito de test se queda sin mapa.

### 3.2 Guardar el split

Se guarda `df_train`/`df_test` ya con `neighbourhood_price_encoded` corregido, pero antes de la imputación y la conversión de booleanas de la sección 2, para que `04_model_training.ipynb` y `05_model_evaluation.ipynb` puedan decidir su propio tratamiento.

In [8]:
df_train.to_csv(f"{PROJECT_DIR}/data/processed/malaga/listings_train.csv", index=False)
df_test.to_csv(f"{PROJECT_DIR}/data/processed/malaga/listings_test.csv", index=False)

## 4. Baseline ingenuo

Un modelo trivial (predecir siempre la media, la mediana, o la mediana por una variable de tamaño) como suelo mínimo: cualquier modelo real tiene que superar esto para que merezca la pena.

### 4.1 Predecir siempre la media

Por definición, un modelo que siempre predice la media del train tiene R² ≈ 0 sobre el test. Sirve como punto cero.

In [9]:
global_mean = y_train.mean()
pred_mean = np.full(len(y_test), global_mean)

print("RMSE:", np.sqrt(mean_squared_error(y_test, pred_mean)))
print("MAE:", mean_absolute_error(y_test, pred_mean))
print("R2:", r2_score(y_test, pred_mean))

RMSE: 234.80852709784423
MAE: 109.30638544449867
R2: -0.0012431821140552746


### 4.2 Predecir siempre la mediana

Dado el sesgo de `price` (skew 7.25 en la EDA), la mediana debería ser un mejor "valor típico" que la media.

In [10]:
global_median = y_train.median()
pred_median = np.full(len(y_test), global_median)

print("RMSE:", np.sqrt(mean_squared_error(y_test, pred_median)))
print("MAE:", mean_absolute_error(y_test, pred_median))
print("R2:", r2_score(y_test, pred_median))

RMSE: 240.44845669329297
MAE: 100.74217838765009
R2: -0.04991908693606861


El MAE mejora (100.74€ frente a 109.31€ con la media), pero el R² empeora (-0.05 frente a ~0.00): el R² compara contra la media por definición, así que cualquier predicción distinta puede bajarlo aunque sea mejor en otros términos. Ambos MAE son bastante más altos que en Valencia (72-74€): Málaga tiene una dispersión de precio real mayor (std 202€ frente a un nivel más moderado en Valencia), con villas de hasta 4775€ todavía dentro del rango tras la limpieza de la EDA.

### 4.3 Predecir la mediana según `accommodates`

`accommodates` es, de las cuatro ciudades, la variable con la correlación lineal más fuerte con `price` (Pearson 0.50, visto en la EDA/feature engineering). Un baseline algo menos ingenuo: mediana por cada valor de `accommodates`, calculada solo con train.

In [11]:
train_medians_by_accommodates = X_train.assign(price=y_train).groupby("accommodates")["price"].median()

pred_accommodates = X_test["accommodates"].map(train_medians_by_accommodates).fillna(global_median)

print("RMSE:", np.sqrt(mean_squared_error(y_test, pred_accommodates)))
print("MAE:", mean_absolute_error(y_test, pred_accommodates))
print("R2:", r2_score(y_test, pred_accommodates))

RMSE: 211.1441257436135
MAE: 82.28805317324185
R2: 0.19040121084675532


Mejora clara: RMSE de 240.4 a 211.1, MAE de 100.7 a 82.3, R² de -0.05 a **0.19**, un listón muy similar al de Valencia (0.21) pese a que aquí `accommodates` tiene una correlación de Pearson bastante más alta. La explicación: en Málaga el propio `price` es más disperso en términos absolutos (la cola de villas), así que agrupar por una sola variable de tamaño explica una fracción similar de esa varianza, aunque en euros los errores absolutos sean mayores.

## 5. Métricas de evaluación

Definir aquí las métricas que se van a usar de forma consistente en todo el modelado (RMSE, MAE, R², MAPE).

### 5.1 Función `evaluate`

Se añade una cuarta métrica, MAPE, el error medio en porcentaje sobre el precio real. Todo se empaqueta en una función para no repetir el código en cada modelo.

In [12]:
def evaluate(y_true, y_pred, name):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    return {"modelo": name, "RMSE": rmse, "MAE": mae, "R2": r2, "MAPE": mape}

### 5.2 Tabla comparativa de los baselines

In [13]:
rows = [
    evaluate(y_test, pred_mean, "Media"),
    evaluate(y_test, pred_median, "Mediana"),
    evaluate(y_test, pred_accommodates, "Mediana por accommodates"),
]

results = pd.DataFrame(rows).set_index("modelo")
results.round(2)

,RMSE,MAE,R2,MAPE
modelo,,,,
Media,234.81,109.31,-0.00,75.00
Mediana,240.45,100.74,-0.05,55.76
Mediana por accommodates,211.14,82.29,0.19,40.12


El MAPE sale alto (40-75%), aunque algo más contenido que en Valencia (45-88%): hay anuncios muy baratos (`price` mínimo 2.10€) donde un error de pocos euros ya es un porcentaje enorme. Conviene fiarse más de RMSE/MAE/R² que del MAPE por sí solo.

## 6. Baseline real: regresión lineal

Un primer modelo simple e interpretable, entrenado sobre `price_log` por el sesgo ya visto en la EDA.

### 6.1 Entrenar

Se entrena sobre `log1p(price)`. Las predicciones se deshacen con `expm1` antes de evaluar.

In [14]:
y_train_log = np.log1p(y_train)

model = LinearRegression()
model.fit(X_train, y_train_log)

pred_log = model.predict(X_test)
pred_lr = np.expm1(pred_log)

/Users/yagocoll/Documents/Master/Airbnb/.venv/lib/python3.9/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/Users/yagocoll/Documents/Master/Airbnb/.venv/lib/python3.9/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: overflow encountered in matmul
  return X @ coef_ + self.intercept_
/Users/yagocoll/Documents/Master/Airbnb/.venv/lib/python3.9/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: invalid value encountered in matmul
  return X @ coef_ + self.intercept_


### 6.2 Evaluar

In [15]:
results.loc["Regresión lineal (log)"] = evaluate(y_test, pred_lr, "Regresión lineal (log)")
results.round(2)

,RMSE,MAE,R2,MAPE
modelo,,,,
Media,234.81,109.31,-0.00,75.00
Mediana,240.45,100.74,-0.05,55.76
Mediana por accommodates,211.14,82.29,0.19,40.12
Regresión lineal (log),178.70,67.36,0.42,26.82


**R²=0.42, MAE≈67.4€**: la mejor de las cuatro ciudades en R² tras Barcelona/Madrid, y notablemente mejor que Valencia (0.32): coherente con que `accommodates` (Pearson 0.50) se acerca bastante más a su propio Spearman (0.61) que en Valencia (0.35 vs. 0.63): la relación tamaño-precio en Málaga es más lineal. Aun así, el MAE en euros (67.4€) es mayor que en Valencia (45.5€): con 74 features, el modelo real explica más varianza relativa, pero la dispersión absoluta de `price` en Málaga (villas incluidas) sigue siendo alta.

### 6.3 Un aviso a tener en cuenta

Al entrenar aparecen avisos de `numpy` (`divide by zero`, `overflow`... `encountered in matmul`), igual que en las otras ciudades.

In [16]:
import numpy.linalg as la

la.cond(X_train.values)

np.float64(2.221448991127767e+19)

Un número de condición altísimo indica una matriz muy mal condicionada: `X` incluye a propósito tanto la versión bruta como la versión `_log` de varias variables (`bedrooms`/`bedrooms_log`...) y varias codificaciones categóricas que se solapan. Aquí se suma además la redundancia ya documentada en `02_feature_engineering.ipynb` (sección 5.2) entre `district_*` y `neighbourhood_price_encoded`, calculados sobre la misma columna. Esto no invalida las métricas (`scikit-learn` resuelve con SVD, sin `NaN`/`inf` en las predicciones), pero sí impide interpretar los coeficientes uno a uno. Se deja igual que en las otras ciudades para `04_model_training.ipynb` (modelo regularizado o selección de variables, si hiciera falta interpretar coeficientes).

## 7. Conclusiones

Resumen de los resultados del baseline.

### Resultados

| Modelo | RMSE | MAE | R² | MAPE |
|---|---|---|---|---|
| Media | 234.81 | 109.31 | -0.00 | 75.00 |
| Mediana | 240.45 | 100.74 | -0.05 | 55.76 |
| Mediana por `accommodates` | 211.14 | 82.29 | 0.19 | 40.12 |
| Regresión lineal (log) | 178.70 | 67.36 | 0.42 | 26.82 |

Cada paso mejora sobre el anterior: agrupar por `accommodates` ya recorta el MAE de 109€ a 82€, y el modelo real con las 74 variables lo baja a 67.4€, con un R² de 0.42: el mejor resultado de un baseline lineal tras Barcelona/Madrid, y muy por delante de Valencia (0.32).

### Tres problemas encontrados y cómo se trataron

- **Fuga de información directa**: `price_log`, `price_per_accommodate` y `price_per_min_night` estaban calculadas a partir de `price` y se habían colado como features. Se excluyeron en la sección 2, igual que en las otras tres ciudades.
- **Fuga más leve, ahora corregida**: `neighbourhood_price_encoded` se calculaba con todo el dataset. Se corrige en la sección 3.1bis, recalculándola solo con `df_train`. Los 11 dummies `district_*`, calculados sobre la misma columna, no tenían este problema por construirse fila a fila.
- **Muestra escasa en `Hotel room`/`Shared room`** (4 y 33 anuncios sobre 8742): el split estratifica por `room_type`.
- **Multicolinealidad severa** en la regresión lineal, por la versión bruta y `_log` de varias variables a la vez, más la redundancia `district_*`/`neighbourhood_price_encoded` propia de Málaga. No afecta a las métricas de predicción, pero impide interpretar los coeficientes uno a uno.

### Una diferencia real con Valencia, no un error de pipeline

El rendimiento del baseline lineal en Málaga (R²=0.42) es notablemente más alto que en Valencia (0.32), aunque el MAE en euros sea mayor. La causa se puede rastrear hasta la EDA y el feature engineering: `accommodates` tiene aquí la correlación de Pearson más fuerte de las cuatro ciudades (0.50) y se acerca bastante a su Spearman (0.61): una relación tamaño-precio más lineal que en Valencia, donde Pearson (0.35) se quedaba muy por detrás de Spearman (0.63). Al mismo tiempo, `price` en Málaga es más disperso en términos absolutos (villas de hasta 4775€ tras la limpieza), lo que mantiene el MAE en euros por encima del de Valencia pese al mejor ajuste relativo. Ambas cosas son coherentes, no contradictorias: R² mide varianza explicada (relativa), MAE mide error típico (absoluto).

### El listón para `04_model_training.ipynb`

Cualquier modelo más complejo (Random Forest, HistGradientBoosting...) tiene que superar claramente **R²=0.42 / MAE≈67.4€**. Un modelo de árboles no necesita la imputación por mediana de la sección 2.3, no le afecta la multicolinealidad de la sección 6.3, y podría beneficiarse de `distance_to_center_km` (sección 8 de `02_feature_engineering.ipynb`), cuya relación con `price` en Málaga es real pero no monótona de forma simple (positiva en bruto, por las villas de las afueras, pero sin ser una tendencia lineal limpia).

### Lo que queda guardado

`listings_train.csv` y `listings_test.csv` en `data/processed/malaga/`, con el mismo split (80/20, `random_state=42`, estratificado por `room_type`) y `neighbourhood_price_encoded` ya corregido, para que `04_model_training.ipynb` y `05_model_evaluation.ipynb` trabajen sobre las mismas filas y los resultados sean comparables entre notebooks.